# Canonical Model 06: Ensemble Uncertainty & IES Diagnostics

This notebook works through the **per-cycle ensemble diagnostics** to look at every time: prior Monte Carlo and prior-data conflict, phi distribution & contribution, parameters at bounds, posterior forecast uncertainty, and **spatial K-field ("property pattern") maps**.

It calibrates a *spatially-varying* K field (one geostatistical parameter per Voronoi cell), so the maps in section 8 answer the workshop's question: *property patterns -- plausible or laughable?* Every plot offers both a `plotly` and a `matplotlib` backend.

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo
from myflopy.modflow.mf6.pest import PestProject
from myflopy.modflow.mf6.grid.plotting import build_choropleth

notebook_header('06', 'Ensemble Uncertainty & IES Diagnostics',
                'Prior MC, prior-data conflict, phi & weight diagnostics, forecast uncertainty, K-field maps.')

## 1. Build and declare the PEST problem (grid K)

In [ ]:
artifact_root = Path('../artifacts/canonical_pest')
artifact_root.mkdir(parents=True, exist_ok=True)

# A SMOOTH synthetic truth K field (a homogeneous base x a smooth anomaly) on
# the two unconfined layers, with a flat-base START -- IES recovers the spatial
# pattern AND the prior brackets the data. (The model's own sharp paleochannel K
# is a near-extreme low-head configuration a smooth prior cannot bracket, which
# is why we use a smooth synthetic anomaly here.)
demo = build_canonical_calibration_demo(artifact_root / 'model', synthetic_k=True,
                                        synthetic_k_base=20.0, start_k_layers=(0, 1))

cal = PestProject(model=demo.model, name='canonical_gridk',
                  workspace=artifact_root / 'template', start_datetime='2024-01-01')
# Grid K: one geostatistically-correlated multiplier per Voronoi cell on each of
# the two unconfined layers. `capture=True` records every realization's resolved
# K field so we can map it. `correlation` is the variogram range (model units):
# the facade draws a *correlated* prior ensemble so realizations are smooth
# fields, not per-cell noise.
cal.parameterize('k', style='grid', layers=[0, 1], correlation=600.0,
                 bounds=(0.05, 20.0), physical=(0.001, 300.0), capture=True)
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0), physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
pst = cal.build('canonical_gridk.pst', noptmax=0)
print('adjustable parameters:', pst.npar_adj, '| nonzero obs:', pst.nnz_obs)

## 2. Prior Monte Carlo -- "early and often"

Evaluate the **prior** parameter ensemble once (`cal.prior(...)`, NOPTMAX = -1) before history matching: is the prior wide enough to explain the data, and is any observation *outside* what the prior can produce? Grey "spaghetti" is the prior simulated ensemble; red triangles are the measured values. **What to look for:** measured points inside the grey band.

In [ ]:
pm = cal.prior(reals=40, workers=12)
display(pm.plot_prior_vs_obs())
display(pm.plot_prior_vs_obs(backend='matplotlib'))

## 3. Prior-data conflict

Where a measured value lies *outside* the prior predictive band, prior and data disagree before calibration starts -- usually a too-narrow prior, a wrong weight, or missing physics. **What to look for:** groups with a high % in conflict; investigate before trusting history matching.

In [ ]:
conflict = pm.conflict()
print('observations in conflict:', int(conflict['in_conflict'].sum()), 'of', len(conflict))
display(pm.plot_conflict())
display(conflict[conflict['in_conflict']].head(10))

## 4. Run PESTPP-IES

A grid-K ensemble (thousands of parameters) with parallel agents -- this is a *go-get-coffee* cell (~minutes). Set `RUN_IES = False` to skip.

In [ ]:
RUN_IES = True
if RUN_IES:
    ies = cal.run_ies(reals=30, iterations=3, workers=12)
    print(ies.settings)
else:
    ies = None
    print('Skipped: set RUN_IES = True to run PESTPP-IES.')

## 5. Phi convergence and distribution (the "pepsi challenge")

`plot_phi` shows the objective function dropping across iterations; `plot_phi_distribution` overlays prior vs posterior phi *histograms* (log scale). **What to look for:** the posterior histogram shifted left of the prior, and a posterior phi near the observation count (a phi far below that is over-fitting).

In [ ]:
if ies is not None:
    display(ies.plot_phi())
    display(ies.plot_phi_distribution())
    display(ies.plot_phi_distribution(backend='matplotlib'))

## 6. Phi contribution by group, and parameters at bounds

`plot_phi_contributions` breaks total phi down by observation group (is one group dominating the misfit?); `parameters_at_bounds` reports the fraction of each parameter group pinned at a bound (are bounds too tight?). **What to look for:** a balanced phi across groups, and few parameters stuck at bounds.

In [ ]:
if ies is not None:
    display(ies.plot_phi_contributions())
    display(ies.parameters_at_bounds())
    display(ies.plot_parameters_at_bounds())

## 7. Forecast uncertainty -- "uncertainty analysis for free"

The posterior distribution of the prediction of interest (a down-valley head near the lake). **What to look for:** history matching should *narrow* the forecast relative to the prior -- but a good fit is not a good prediction, so value the posterior spread.

In [ ]:
if ies is not None:
    display(ies.forecasts())
    name = ies.forecast_names[0]
    display(ies.forecast(name).plot())

## 8. Property patterns -- did IES recover the K field?

*"Property patterns: plausible or laughable?"* Because we declared `capture=True`, every realization's resolved K field is recorded, so we can map it. Compare the known **truth** K against the **posterior mean** (what IES recovered) and the **posterior std** (where K is still uncertain).

**What to look for:** with only head observations, fine K structure *cannot* be fully resolved -- the posterior mean is a smooth, plausible field that adjusts K **near the observation wells** and stays close to the prior elsewhere. The std map shows exactly where the data does (and does not) constrain K. A posterior that reproduced the sharp truth everywhere from sparse heads would be *too good to be true*.

In [ ]:
if ies is not None:
    layer = 0
    truth_k = list(demo.truth_k[layer])                       # known truth (layer 1)
    display(build_choropleth(demo.model.vor, custom_zs=truth_k, layer=layer))
    display(ies.plot_field('k', stat='mean', which='posterior', layer=layer))   # recovered
    display(ies.plot_field('k', stat='std', layer=layer))                       # uncertainty
    display(ies.plot_field('k', stat='change', layer=layer, backend='matplotlib'))  # prior->post
    display(ies.field('k', layer=layer).head())

## 9. The single best realization, and the report bundle

Carry the **base** (minimum-error-variance) realization forward, never the lowest-phi one. `report(...)` bundles the diagnostics into one HTML file.

In [ ]:
if ies is not None:
    print('recommended realization:', ies.best())
    report_path = ies.report(artifact_root / 'uncertainty_review.html')
    print('wrote', report_path)

## Interpretation checklist (run this every cycle)

- **Prior Monte Carlo first:** is the prior wide enough, and is any observation in conflict with it?
- **Phi:** did it drop, and is the posterior phi near (not far below) the observation count?
- **Phi by group / parameters at bounds:** is one group dominating, or are bounds too tight?
- **Forecast:** value the posterior *spread* -- a good fit is not a good prediction.
- **Property patterns:** are the posterior K maps geologically plausible? Trust them only where the data constrains them (low posterior std).
- Carry the **base** realization forward.